# ECM Client Setup for Google Colab

This notebook sets up the ECM factorization client on Google Colab.

**Repository:** https://github.com/kyleaskine/ecm-wrapper

## ⚠️ IMPORTANT: Enable GPU Runtime FIRST!

**Before running any cells**, ensure you have a GPU runtime:

1. Click **Runtime** menu → **Change runtime type**
2. Set **Hardware accelerator** to **T4 GPU**
3. Click **Save**

This will disconnect your current session - that's expected!

## Instructions

1. **FIRST**: Enable T4 GPU runtime (see above)
2. Run the **GPU Check Cell** to verify GPU is available
3. Run the **Setup Cell** and enter your username
4. Use the **Test Cell** to verify everything works
5. Use the **Auto-Work Cell** to get work from the server and submit results (both stages on the GPU)

**Note:** You'll need to re-run setup each time you start a new Colab session.

In [ ]:
# ============================================================
# GPU CHECK - Run this FIRST!
# ============================================================

import subprocess
import sys

print("=" * 60)
print("  GPU Runtime Check")
print("=" * 60)
print()

# Check for NVIDIA GPU
try:
    result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                          capture_output=True, text=True, timeout=10)
    if result.returncode == 0 and result.stdout.strip():
        gpu_info = result.stdout.strip()
        print(f"✅ GPU DETECTED: {gpu_info}")
        print()
        print("You're good to go! Proceed to the Setup Cell below.")
    else:
        raise Exception("No GPU found")
except Exception as e:
    print("❌ NO GPU RUNTIME DETECTED!")
    print()
    print("You need to enable GPU runtime before continuing:")
    print()
    print("1. Click 'Runtime' menu at the top")
    print("2. Select 'Change runtime type'")
    print("3. Set 'Hardware accelerator' to 'T4 GPU'")
    print("4. Click 'Save'")
    print()
    print("⚠️  This will disconnect your session - reconnect and run this cell again.")
    print()
    sys.exit(1)  # Stop execution here

print("=" * 60)

In [ ]:
# ============================================================
# SETUP CELL - Run this first!
# ============================================================

import os
import subprocess

print("=" * 60)
print("  ECM Factorization Client - Google Colab Setup")
print("=" * 60)
print()

# ============================================================
# Step 1: User Configuration
# ============================================================
USERNAME = input("📝 Enter your username: ").strip()

PRIORITY_VALUE = input("⭐ Enter priority filter (default: 5): ").strip()
PRIORITY_VALUE = PRIORITY_VALUE if PRIORITY_VALUE else "5"

# Production API endpoint
API_ENDPOINT = "https://ecm.kyleaskine.com/api/v1"

print()
print("=" * 60)

# ============================================================
# Step 2: Clone Repository
# ============================================================
print("📦 Cloning ecm-wrapper repository...")
if os.path.exists("/content/ecm-wrapper"):
    !rm -rf /content/ecm-wrapper
!git clone -q https://github.com/kyleaskine/ecm-wrapper.git /content/ecm-wrapper
os.chdir("/content/ecm-wrapper/client")
print("✓ Repository cloned")

# ============================================================
# Step 3: Create data directory
# ============================================================
print("📁 Creating data directory...")
!mkdir -p data
print("✓ Data directory created")

# ============================================================
# Step 4: Get the ECM Binary
# ============================================================
# GMP-ECM from github.com/kyleaskine/gmp-ecm (GPU stage 1 and GPU stage 2),
# built for Colab's GPU and CUDA version. If it can't be downloaded, or fails
# the GPU test on this runtime (e.g. Colab changed its CUDA version), it is
# built from source here instead. That is slow: allow 20-40 minutes.
# Set FORCE_BUILD = True to always build from source.
import shutil
FORCE_BUILD = False
ECM_DOWNLOAD_URL = "https://ecm.kyleaskine.com/downloads/ecm/ecm-colab.gz"
ECM_PATH = "/content/ecm"
ECM_SRC = "/content/gmp-ecm"
# A C50 with no factor found at B1=1000: only checks that stage 1 and stage 2
# both run on the GPU without an error.
TEST_N = "10000000000000000000000083000000000000000000000091"

def ecm_gpu_works(path):
    if not os.path.exists(path):
        return False
    try:
        r = subprocess.run([path, "-gpu", "-gpucurves", "64", "1000"], input=TEST_N + "\n",
                           capture_output=True, text=True, timeout=300)
    except Exception as e:
        print(f"   GPU test could not run: {e}")
        return False
    out = r.stdout + r.stderr
    if "Step 2 took" in out and "of GPU time" in out and "Error" not in out:
        return True
    print("   GPU test output:")
    print("   " + "\n   ".join(out.strip().splitlines()[-10:]))
    return False

def build_ecm():
    cap = subprocess.run(["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.split()
    arch = cap[0].replace(".", "") if cap else ""
    print(f"🔨 Building GMP-ECM for sm_{arch or '(all)'} - this takes a while...")
    subprocess.run("apt-get -qq update && apt-get -qq install -y autoconf automake libtool m4 libgmp-dev",
                   shell=True, check=True, stdout=subprocess.DEVNULL)
    shutil.rmtree(ECM_SRC, ignore_errors=True)
    subprocess.run(["git", "clone", "-q", "--depth", "1",
                    "https://github.com/kyleaskine/gmp-ecm.git", ECM_SRC], check=True)
    gpu_flag = f"--enable-gpu={arch}" if arch else "--enable-gpu"
    r = subprocess.run(f"autoreconf -i && ./configure {gpu_flag} --with-cuda=/usr/local/cuda "
                       f"&& make -j{os.cpu_count() or 2}",
                       shell=True, cwd=ECM_SRC, capture_output=True, text=True)
    with open(f"{ECM_SRC}/build.log", "w") as f:
        f.write(r.stdout + r.stderr)
    if r.returncode != 0:
        print("❌ Build failed. Last lines of the log:")
        print("\n".join((r.stdout + r.stderr).strip().splitlines()[-25:]))
        raise RuntimeError("GMP-ECM build failed")
    shutil.copy(f"{ECM_SRC}/ecm", ECM_PATH)
    os.chmod(ECM_PATH, 0o755)
    print("✓ GMP-ECM built")

ecm_ok = False
if not FORCE_BUILD:
    print("⬇️  Downloading ECM binary (built for Colab)...")
    dl = subprocess.run(["wget", "-q", ECM_DOWNLOAD_URL, "-O", f"{ECM_PATH}.gz"])
    if dl.returncode == 0 and subprocess.run(["gunzip", "-f", f"{ECM_PATH}.gz"]).returncode == 0:
        os.chmod(ECM_PATH, 0o755)
        ecm_ok = ecm_gpu_works(ECM_PATH)
        if not ecm_ok:
            print("⚠️  The downloaded binary failed the GPU test on this runtime.")
    else:
        print(f"⚠️  Could not download {ECM_DOWNLOAD_URL}")
if not ecm_ok:
    build_ecm()
    ecm_ok = ecm_gpu_works(ECM_PATH)
    if not ecm_ok:
        raise RuntimeError("The GMP-ECM build failed the GPU test (see output above)")
    print(f"   To publish it as the Colab binary: gzip -c {ECM_PATH} > ecm-colab.gz, then put it")
    print("   at /var/www/ecm-downloads/ecm/ecm-colab.gz on the server.")
print("✓ ECM binary works (GPU stage 1 and stage 2)")

# ============================================================
# Step 5: Install Python Dependencies
# ============================================================
print("📚 Installing Python dependencies...")
!pip install -q requests pyyaml
print("✓ Dependencies installed (requests, pyyaml)")

# ============================================================
# Step 6: Create client.local.yaml
# ============================================================
print("⚙️  Creating client.local.yaml...")

local_config = f"""# Google Colab Local Configuration
# This file overrides settings in client.yaml

api:
  endpoint: "{API_ENDPOINT}"
  timeout: 30

client:
  username: "{USERNAME}"
  cpu_name: "Google-Colab"

programs:
  gmp_ecm:
    path: "{ECM_PATH}"
    gpu_enabled: true
    gpu_device: 0

# Logging configuration
logging:
  level: "INFO"
  file: "data/logs/ecm_client.log"
  console: true
"""

with open("client.local.yaml", "w") as f:
    f.write(local_config)

print("✓ Configuration file created")

# ============================================================
# Step 7: Setup Complete - Display Summary
# ============================================================
print()
print("=" * 60)
print("✅ SETUP COMPLETE!")
print("=" * 60)
print(f"Username:      {USERNAME}")
print(f"API Endpoint:  {API_ENDPOINT}")
print(f"CPU Name:      Google-Colab")
print(f"ECM Binary:    {ECM_PATH}")
print(f"Working Dir:   /content/ecm-wrapper/client")
print(f"GPU:           Enabled (device 0)")
print(f"Priority:      {PRIORITY_VALUE}")
print("=" * 60)
print()
print("🚀 Ready! Run the cells below to start factoring.")
print()
print("💡 TIP: You can upload numbers.txt to the data/ directory")
print("   using the file browser on the left, then run batch scripts.")
print()

## Test Cell

Run this to verify ECM is working correctly (no API submission).

In [ ]:
# Test factorization (no API submission - this is the default for ecm_wrapper.py)
print("Testing ECM binary...")
print("-" * 60)
!python3 ecm_wrapper.py --composite "123456789012345" --curves 5 --b1 11000
print("-" * 60)
print("Test complete! Check output above for results.")

## Auto-Work Mode (GPU, both stages)

This will continuously get work from the server and submit results. Each assignment is one GPU batch with both ECM stages, at the B1 that fits the composite's current t-level; then it moves on to the next composite, so stopping the session loses at most one batch.

In [ ]:
# Auto-work mode: one GPU batch (both stages) per composite, then the next one
print(f"Running auto-work with Priority={PRIORITY_VALUE} (GPU, both stages, one batch per composite)")
print("-" * 60)
!python3 ecm_client.py --standard --gpu --one-batch --priority {PRIORITY_VALUE} -v

## Alternative: Stage 1 Only (Progressive Strategy)

This will continuously get work from the server (least ECM done first) and submit results.

Runs stage 1 only on the GPU and uploads the residues, for other clients to run stage 2.

In [ ]:
# Auto-work mode with progressive strategy (Stage 1 only)
# Processes composites with the least ECM work done first
print(f"Running auto-work (progressive) with Priority={PRIORITY_VALUE} (Stage 1 only)")
print("-" * 60)
!python3 ecm_client.py --work-type progressive --stage1-only --priority {PRIORITY_VALUE} -v

## Alternative: Run Auto-Work with Limited Count

Process a specific number of work assignments then stop.

In [ ]:
# Process 5 work assignments (one GPU batch each) then stop
!python3 ecm_client.py --standard --gpu --one-batch --work-count 5 --priority {PRIORITY_VALUE} -v

## View Logs

In [ ]:
# View recent log entries
!tail -n 50 data/logs/ecm_client.log

## View Configuration

In [ ]:
# Display current configuration
!cat client.local.yaml

## View Factors Found

In [ ]:
# Display any factors that have been found
!cat data/factors_found.txt 2>/dev/null || echo "No factors found yet."